# SprintShield Analytics - Sprint Capacity Audit

**Quantifying unplanned production support demand and forecasting delivery confidence.**

This notebook walks the full pipeline end-to-end:

1. Load (or synthesise) multi-tower sprint telemetry
2. Executive KPIs - Support Tax, Flow Efficiency, Predictability, MTTR
3. Stochastic demand modelling - dispersion index, Poisson vs. Negative Binomial
4. Resolution-effort fitting - Log-Normal vs. Weibull
5. Monte Carlo sprint-capacity simulation (10,000 runs)
6. What-if planning and buffer sizing
7. Root-cause analysis of the assets draining developer hours


## 0. Setup


In [ ]:
# --- environment bootstrap ------------------------------------------------- #
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if (ROOT / "src").exists() is False and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


## 1. Load telemetry

Reads the configured storage backend; falls back to the synthetic generator when the lakehouse is empty.


In [ ]:
from sprintshield.analytics.metrics import executive_summary, tower_metrics_table, asset_metrics_table
from sprintshield.client.synthetic_generator import generate_telemetry
from sprintshield.config import load_config
from sprintshield.storage.factory import get_storage_adapter

CONFIG = load_config(project_root=ROOT)
adapter = get_storage_adapter(CONFIG)
snapshots = adapter.get_sprint_history()
tickets = adapter.get_support_tickets()

if snapshots.empty:
    snapshots, tickets = generate_telemetry(sprints=24, config=CONFIG)

print(f'backend   : {adapter.backend}')
print(f'towers    : {", ".join(CONFIG.tower_ids())}')
print(f'snapshots : {snapshots.shape} | tickets: {tickets.shape}')
snapshots.head()

## 2. Executive KPIs


In [ ]:
NAMES = {t.tower_id: t.name for t in CONFIG.towers}
summary = executive_summary(snapshots, tickets, NAMES)
pd.Series(summary).to_frame('value')

In [ ]:
tower_metrics_table(snapshots, tickets)

## 3. Stochastic demand modelling

The **dispersion index** `D = sigma^2 / mu` decides the arrival family: `D ~ 1` -> Poisson, `D > 1.2` -> Negative Binomial (batch release overdispersion).


In [ ]:
from sprintshield.analytics.arrival_model import compare_arrival_models, dispersion_index

rows = []
for tower_id, group in snapshots.groupby('tower_id'):
    counts = pd.to_numeric(group['incident_count'], errors='coerce').dropna().tolist()
    best = compare_arrival_models(counts, CONFIG.arrival_model).iloc[0]
    rows.append({
        'tower_id': tower_id,
        'dispersion': round(dispersion_index(counts), 3),
        'selected': best['family'],
        'fitted_mean': round(float(best['fitted_mean']), 2),
        'observed_mean': round(float(best['observed_mean']), 2),
        'aic': round(float(best['aic']), 2),
    })
pd.DataFrame(rows)

## 4. Resolution-effort distributions


In [ ]:
from sprintshield.analytics.effort_distribution import (
    compare_effort_models,
    effort_fits_to_frame,
    fit_effort_by_group,
)

fits = fit_effort_by_group(tickets, 'tower_id', config=CONFIG.effort_model)
effort_fits_to_frame(fits, key_name='tower_id')[
    ['tower_id', 'family', 'param_mu', 'param_sigma', 'mean', 'std', 'sample_size']
]

In [ ]:
compare_effort_models(
    tickets.loc[tickets['tower_id'] == 'DE', 'resolution_hours'].dropna().tolist()
)

## 5. Monte Carlo sprint-capacity simulation

10,000 vectorised runs per tower:

1. `N ~ NegBinom(r, p)` - incoming incidents
2. `E_i ~ LogNormal(mu, sigma)` - resolution effort per incident
3. `H_support = sum E_i` - unplanned overhead
4. `H_net = H_gross - H_support` - net delivery capacity


In [ ]:
from sprintshield.analytics.monte_carlo import simulate_all_towers

simulations, simulation_summary = simulate_all_towers(CONFIG, snapshots, tickets, runs=10_000)
simulation_summary[[
    'tower_id', 'mean_incidents', 'mean_support_hours', 'mean_support_tax',
    'p85_support_hours', 'delivery_confidence', 'max_committed_hours_at_target',
]].round(3)

In [ ]:
from sprintshield.dashboard.views import build_confidence_curve, build_support_tax_bar, build_tower_radar

build_confidence_curve(simulations['DE'], planned_hours=240.0, target_confidence=0.85)

In [ ]:
build_support_tax_bar(tower_metrics_table(snapshots, tickets), NAMES)

In [ ]:
build_tower_radar(tower_metrics_table(snapshots, tickets))

## 6. What-if planning

Change `TOWER` and `PLANNED_POINTS` and re-run the cell.


In [ ]:
from sprintshield.analytics.monte_carlo import MonteCarloSprintSimulator

TOWER = 'DE'
PLANNED_POINTS = 80

simulator = MonteCarloSprintSimulator.from_tower(CONFIG, TOWER)
result = simulator.simulate(
    gross_hours=CONFIG.tower(TOWER).gross_hours_per_sprint,
    planned_hours=PLANNED_POINTS * CONFIG.simulation.hours_per_point,
)
pd.Series(result.summary()).to_frame('value')

In [ ]:
print(f'Safe commitment @85% : {result.max_committed_hours(0.85) / CONFIG.simulation.hours_per_point:.1f} pts')
print(f'Support buffer       : {result.buffer_hours(0.85):.1f} h')
result.confidence_curve().iloc[::6]

## 7. Root cause - assets draining developer hours


In [ ]:
asset_metrics_table(tickets).head(12)

## 8. Active support WIP


In [ ]:
open_wip = tickets[tickets['status'] != 'Done']
print(f'Open blockers: {len(open_wip)}')
open_wip.sort_values('priority')[
    ['ticket_id', 'tower_id', 'asset', 'priority', 'issue_type', 'summary']
].head(20)